# **Day 60: itertools and functools**

These two modules have many small but very useful tools. This is the last day of Phase 3.

## **itertools: Endless Iterators**

In [1]:
from itertools import count, cycle, repeat, islice

print(list(islice(count(700001), 5)))                  # Counting forever (we take only 5)
print(list(islice(cycle(["Mon", "Tue", "Wed"]), 7)))    # Repeat a list forever
print(list(repeat("Dhak", 3)))

[700001, 700002, 700003, 700004, 700005]
['Mon', 'Tue', 'Wed', 'Mon', 'Tue', 'Wed', 'Mon']
['Dhak', 'Dhak', 'Dhak']


## **Combinations and Permutations**
- **Permutations:** Order **matters** (A-B is different from B-A).
- **Combinations:** Order **does not matter** (A-B is the same as B-A).

In [2]:
from itertools import permutations, combinations, product

teams = ["MB", "EB", "MSC"]
print(list(permutations(teams, 2)))       # Home and away matches
print(list(combinations(teams, 2)))       # Just pairs
print(list(product(["veg", "chicken"], ["roll", "biryani"])))

[('MB', 'EB'), ('MB', 'MSC'), ('EB', 'MB'), ('EB', 'MSC'), ('MSC', 'MB'), ('MSC', 'EB')]
[('MB', 'EB'), ('MB', 'MSC'), ('EB', 'MSC')]
[('veg', 'roll'), ('veg', 'biryani'), ('chicken', 'roll'), ('chicken', 'biryani')]


## **Example from Remote Sensing: All Possible Band Pairs**
Researchers often test **every pair of bands** to find which "normalized difference index" works best for a job, like finding water or buildings. `combinations` gives us every pair without repeating.

In [3]:
bands = {"Blue": 0.06, "Green": 0.09, "Red": 0.07, "NIR": 0.38, "SWIR1": 0.22}   # One pixel's values

print("Normalized difference for every band pair:")
for (b1, v1), (b2, v2) in combinations(bands.items(), 2):
    nd = (v1 - v2) / (v1 + v2)
    print(f"  ({b1:<5} - {b2:<5}) / ({b1} + {b2}) = {nd:+.3f}")

Normalized difference for every band pair:
  (Blue  - Green) / (Blue + Green) = -0.200
  (Blue  - Red  ) / (Blue + Red) = -0.077
  (Blue  - NIR  ) / (Blue + NIR) = -0.727
  (Blue  - SWIR1) / (Blue + SWIR1) = -0.571
  (Green - Red  ) / (Green + Red) = +0.125
  (Green - NIR  ) / (Green + NIR) = -0.617
  (Green - SWIR1) / (Green + SWIR1) = -0.419
  (Red   - NIR  ) / (Red + NIR) = -0.689
  (Red   - SWIR1) / (Red + SWIR1) = -0.517
  (NIR   - SWIR1) / (NIR + SWIR1) = +0.267


Some of these pairs are famous indices:
- **(NIR - Red) / (NIR + Red)** = NDVI (vegetation). It appears as (Red - NIR), which is just the same value with the opposite sign.
- **(Green - NIR) / (Green + NIR)** = NDWI (water)
- **(SWIR1 - NIR) / (SWIR1 + NIR)** = NDBI (built-up area)

## **chain, accumulate, groupby, pairwise**

In [4]:
from itertools import chain, accumulate, groupby, pairwise

print(list(chain(["Howrah"], ["Sealdah", "Shalimar"], ("KGP",))))     # Join many lists

daily_rain = [0, 12, 30, 5, 0, 44]
print(list(accumulate(daily_rain)))                                    # Running total

stops = ["Howrah", "Santragachi", "Mecheda", "Kharagpur"]
for a, b in pairwise(stops):                                           # Each pair of neighbours
    print(f"{a} -> {b}")

['Howrah', 'Sealdah', 'Shalimar', 'KGP']
[0, 12, 42, 47, 47, 91]
Howrah -> Santragachi
Santragachi -> Mecheda
Mecheda -> Kharagpur


In [5]:
# groupby needs the data to be SORTED by the key first
people = [("Kolkata", "Ria"), ("Siliguri", "Jit"), ("Kolkata", "Sam"), ("Siliguri", "Tia"), ("Digha", "Om")]
people.sort(key=lambda p: p[0])
for city, group in groupby(people, key=lambda p: p[0]):
    print(city, [name for _, name in group])

Digha ['Om']
Kolkata ['Ria', 'Sam']
Siliguri ['Jit', 'Tia']


## **Example from GIS: Length of a Route Using pairwise**
A route is a list of points. Its total length is the sum of the distances between each **pair of neighbouring points**. `pairwise` gives us these pairs directly.

In [6]:
import math

def haversine(p1, p2):
    lat1, lon1 = map(math.radians, p1)
    lat2, lon2 = map(math.radians, p2)
    a = math.sin((lat2 - lat1) / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(a))

route = [(22.5726, 88.3639), (22.3500, 88.1000), (22.3000, 87.9200), (21.9000, 87.7800), (21.6266, 87.5074)]
legs = [haversine(a, b) for a, b in pairwise(route)]
print("Each leg (km):", [round(x, 1) for x in legs])
print("Total route length:", round(sum(legs), 1), "km")
print("Running distance:", [round(x, 1) for x in accumulate(legs)])

Each leg (km): [36.7, 19.3, 46.8, 41.4]
Total route length: 144.2 km
Running distance: [36.7, 56.0, 102.8, 144.2]


## **functools: partial**
`partial` makes a new function with some inputs **already filled in**.

In [7]:
from functools import partial, reduce, cache

def fare(km, rate, base):
    return base + km * rate

taxi_fare = partial(fare, rate=20, base=30)
toto_fare = partial(fare, rate=8, base=10)
print(taxi_fare(6), toto_fare(6))

from_binary = partial(int, base=2)
print(from_binary("101101"))

150 58
45


## **functools: cache**
`@cache` remembers the results of a function, so it does not calculate the same thing twice.

In [8]:
@cache
def grid_paths(r, c):
    if r == 0 or c == 0:
        return 1
    return grid_paths(r - 1, c) + grid_paths(r, c - 1)

print(grid_paths(16, 16))

601080390


## **Summary of Phase 3 (Day 41 to Day 60)**
| Topic | Days |
|---|---|
| Classes, objects, constructor, self, magic methods | 41 to 43 |
| Encapsulation, inheritance, polymorphism, abstraction | 44 to 50 |
| Dataclasses and enums | 51, 52 |
| Iterators and generators | 54, 55 |
| Decorators and context managers | 56 to 58 |
| collections, itertools, functools | 59, 60 |

In Phase 4, we move to **advanced Python**: type hints, regular expressions, testing, working in parallel, APIs and more.

In [9]:
# Quick question
# How many items will this give?
# from itertools import combinations
# print(len(list(combinations(range(5), 2))))